In [1]:
import re

#import polars as pl
import pandas as pd
import numpy as np

In [2]:
sample_names = ["HG001", "HG002", "HG003", "HG004"]

In [3]:

# function that reads final tsv file into polars lazyframe
def tsv_to_df(tsv_path):
    sample_column_names = ["CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "SEQ_CONTEXT"]
    df = pl.scan_csv(tsv_path, has_header=False, separator='\t', new_columns=sample_column_names)
    return df

# function to filter dataframe for snps only
def filter_snps(df):
    return df.filter((pl.col("REF").str.len_bytes() == 1) & (pl.col("ALT").str.len_bytes() == 1))

# function that reads giab final tsv file into polars lazyframe
def giab_tsv_to_df(tsv_path):
    giab_column_names = ["CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "HG00X", "FORMAT", "INFO", "SEQ_CONTEXT"]
    df = pl.scan_csv(tsv_path, has_header=False, separator='\t', new_columns=giab_column_names)
    df = df.drop(["HG00X", "FORMAT"])
    df = filter_snps(df)
    return df

# function that labels the dataset as true variant or artifact, takes a list of sample names
def label_datasets(sample_names):
    artifacts = []
    true_variants = []
    for sample_name in sample_names:
        sample_df = tsv_to_df(f'/home/dzhou/nanopore/output/final_tsv/{sample_name}_merged_realigned_highconf_filtered_altcov_2_with_seq.tsv')
        giab_df = giab_tsv_to_df(f'/home/dzhou/nanopore/output/final_tsv/{sample_name}_GRCh38_1_22_v4.2.1_benchmark_highconf_with_seq.tsv')
        
        true_variants_df = sample_df.join(giab_df, on=["CHROM", "POS", "REF", "ALT"], how='semi')
        true_variants.append(true_variants_df)
        
        artifacts_df = sample_df.join(giab_df, on=["CHROM", "POS", "REF", "ALT"], how='anti')
        artifacts.append(artifacts_df)
        
        giab_artifacts_df = giab_df.join(sample_df, on=["CHROM", "POS", "REF", "ALT"], how='anti')
        artifacts.append(giab_artifacts_df)

        ##TEST SECTION
        '''
        print(sample_name)
        print("true variants:", true_variants_df.select(pl.len()).collect().row(0)[0])
        print("artifacts:", artifacts_df.select(pl.len()).collect().row(0)[0])
        print("giab_artifacts:", giab_artifacts_df.select(pl.len()).collect().row(0)[0])
        '''
    
    # concatenate total artifacts/variants across all datasets
    total_artifacts_df = pl.concat(artifacts, how="vertical_relaxed")
    total_true_variants_df = pl.concat(true_variants, how="vertical_relaxed")

    # adding column to label true variant or artifact
    labeled_total_artifacts_df = total_artifacts_df.with_columns(TRUE_VARIANT = False)
    labeled_total_true_variants_df = total_true_variants_df.with_columns(TRUE_VARIANT = True)
    
    # combine artifacts and true variants into one df
    result_df = pl.concat([labeled_total_artifacts_df, labeled_total_true_variants_df], how="vertical_relaxed")
    
    # deduplication step
    deduped_result_df = result_df.unique(subset=["CHROM", "POS", "REF", "ALT", "TRUE_VARIANT"])
    
    return deduped_result_df




In [4]:
#%%time
#result_df = label_datasets(sample_names)
#result_df.collect().write_parquet("/home/dzhou/nanopore/result_df.parquet")

In [35]:
# splits INFO column to extract variant features in df
def extract_value(field, info):
    value = info.split(field + '=')[1].split(';')[0]
    return float(value)

# uses extract_value to map across whole df
def parse_info_col(df):
    for field in ['DP','AF','SB']:
        df[field] = df['INFO'].apply(lambda x: extract_value(field, x))

    # remove INFO and ID column
    df.pop("INFO")
    df.pop("ID")
    
    # rearrange remaining column order appropriately
    df = df[['CHROM', 'POS', 'REF', 'ALT', 'QUAL', 'FILTER', 'DP', 'AF', 'SB', 'GC', 'SEQ_CONTEXT', 'TRUE_VARIANT']]
    return df


def filter_and_process_data(df):
    # filter sequences with N's
    df = df[~df['SEQ_CONTEXT'].str.contains('[^ACTG]', regex=True)]
    
    # filters out variants that weren't called in the sample data (only 294)
    df = df[df['INFO'].str.contains("DP=")]

    # filters out sequences with length != 201
    df = df[df['SEQ_CONTEXT'].str.len() == 201]

    # convert bool to float for target variable
    df['TRUE_VARIANT'] = df['TRUE_VARIANT'].astype(float)
    
    # add feature for GC content
    df['GC'] = df['SEQ_CONTEXT'].apply(lambda x: (x.count('G') + x.count('C'))/len(x))

    return df

In [26]:
sampled_df = pd.read_parquet("/home/dzhou/nanopore/parquets/sampled_result_df.parquet")

In [36]:
%%time
clean_df = filter_and_process_data(sampled_df)
filtered_df_final = parse_info_col(clean_df)
#filtered_df_final.to_parquet('/home/dzhou/nanopore/parquets/sampled_parsed_result_df.parquet')

CPU times: user 1min 46s, sys: 11 s, total: 1min 57s
Wall time: 1min 54s


,CHROM,POS,REF,ALT,QUAL,FILTER,DP,AF,SB,GC,SEQ_CONTEXT,TRUE_VARIANT
0,chr1,24933589,A,G,2,PASS,45.0,0.044444,6.0,0.557214,GTGAAATGTCCAGTTCTCATATTCAAAGCTTACTAGGACTCCAAGC...,0.0
1,chr1,75049396,A,G,8,PASS,73.0,0.041096,2.0,0.363184,AAAAATCAAAACGATGGAACTCATGGACTTAGACAGTAGAAGGATA...,0.0
2,chr1,167136770,G,A,1,PASS,114.0,0.017544,3.0,0.422886,ATTCCATTTAGAAAGCTGATTGACCACCTCTGCTGAACAATACAGC...,0.0
3,chr1,214108145,T,C,2,PASS,83.0,0.024096,3.0,0.512438,CCCCCACCTCCTCCCAGAGCCAGGTGAGGTTGAGAGCAGAGAAAAT...,0.0
4,chr1,118669748,G,T,0,PASS,113.0,0.017699,3.0,0.303483,TGTAGAATTCGATAAGCTGATTTTAAAATTTATGTGAGAAAGCAAA...,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
19071828,chr9,137036886,C,G,360,PASS,44.0,0.431818,2.0,0.592040,AAAAAAAAAAACAATTCCAAAAAGGCGAGTGGGGAGCAGGGTGCTA...,1.0
19071829,chr9,137226375,C,G,461,PASS,48.0,0.395833,2.0,0.562189,AAGGCCCAGGCAATAAAGCAGGGTGATCTTCCTCCCAGCAATTCTT...,1.0
19071830,chr9,137844138,T,C,1571,PASS,45.0,0.977778,0.0,0.567164,TCAGGTCGCTCCCCGCCAGGCTGAATCAGGCTCCAGCTCTTCTTCA...,1.0
19071831,chr9,137929488,G,A,468,PASS,57.0,0.421053,1.0,0.472637,TACAAAAACATTAAAAAGTTAGCCAGGCATGGTGGTGTGTGCCTGT...,1.0
